In [1]:
import os
import re
import joblib
import pickle
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import zscore

# home-grown
import utils as ut

In [110]:
from sklearn.linear_model import LogisticRegression
import statsmodels.api as sm

In [3]:
pd.options.display.max_columns = 60

In [4]:
%load_ext autoreload
%autoreload 2

In [5]:
df_mm, dict_colnames = ut.load_mm_dataset("med_seq")

In [6]:
df_mm[dict_colnames["l_colnames_single_zscale"]] = df_mm[dict_colnames["l_colnames_single_zscale"]].apply(zscore)

In [248]:
cols_subset = ["left_man", "left_woman", "left_boy", "left_girl", "left_oldman", "left_oldwoman",
    "right_man", "right_woman", "right_boy", "right_girl", "right_oldman", "right_oldwoman",
    "Eastern", "Southern"]
df_subset = df_mm[cols_subset + ["right_picked"]].copy()

In [249]:
df_subset["is_train"] = np.random.choice([True, False], df_subset.shape[0], p=[.8, .2])

In [250]:
df_subset["oldmen_rml"] = df_subset["right_oldman"] - df_subset["left_oldman"]
df_subset["oldwomen_rml"] = df_subset["right_oldwoman"] - df_subset["left_oldwoman"]
df_subset["boys_rml"] = df_subset["right_boy"] - df_subset["left_boy"]
df_subset["girls_rml"] = df_subset["right_girl"] - df_subset["left_girl"]
df_subset["men_rml"] = df_subset["right_man"] - df_subset["left_man"]
df_subset["women_rml"] = df_subset["right_woman"] - df_subset["left_woman"]
df_subset["Eastern"] = df_subset["Eastern"] - .5
df_subset["Southern"] = df_subset["Southern"] - .5

In [251]:
cols_ia = [c for c in df_subset.columns.to_list() if re.search("_rml", c)]

In [252]:
for c_ia in cols_ia:
    df_subset[f"""{c_ia}xEastern"""] = df_subset[c_ia] * df_subset["Eastern"]
    df_subset[f"""{c_ia}xSouthern"""] = df_subset[c_ia] * df_subset["Southern"]

In [253]:
df_subset["oldmen_rmlxEastern_vs_Southern"] = df_subset["oldmen_rmlxEastern"] - df_subset["oldmen_rmlxSouthern"]
df_subset["girls_rmlxEastern_vs_Southern"] = df_subset["girls_rmlxEastern"] - df_subset["girls_rmlxSouthern"]

df_subset = df_subset.drop(columns=["oldmen_rmlxEastern", "oldmen_rmlxSouthern", "girls_rmlxEastern", "girls_rmlxSouthern"])

In [254]:
cols_ia = [c for c in df_subset.columns.to_list() if re.search("rml", c)]

In [255]:
cols_subset = cols_ia +  ["Eastern", "Southern"]

take a subset of the data
at 100k, no overfit on train anymore, and accuracy train ~ accuracy test

In [256]:
n_subset = 200000

In [257]:
X_train = df_subset.query("is_train")[cols_subset].head(n_subset).to_numpy()
X_train = sm.add_constant(X_train)
X_test = df_subset.query("~is_train")[cols_subset].head(n_subset).to_numpy()
X_test = sm.add_constant(X_test)
y_train = np.ravel(df_subset.query("is_train")["right_picked"].head(n_subset).to_numpy())
y_test = np.ravel(df_subset.query("~is_train")["right_picked"].head(n_subset).to_numpy())

In [258]:
fit_model = True

In [259]:
model_full = sm.Logit(y_train, X_train)

if fit_model:
    result = model_full.fit()
    joblib.dump(result, "models/mm-sklearn/logreg-age-culture.pkl")
else:
    result = joblib.load("models/mm-sklearn/logreg-age-culture.pkl")

Optimization terminated successfully.
         Current function value: 0.657722
         Iterations 5


In [260]:
y_test_pred = (result.predict(X_test) > .5).astype(int)
y_train_pred = (result.predict(X_train) > .5).astype(int)
df_train_preds_full = pd.DataFrame({"y_train_true":y_train, "y_train_pred":y_train_pred}) 
df_train_preds_full["is_correct"] = df_train_preds_full["y_train_true"] == df_train_preds_full["y_train_pred"]
df_test_preds_full = pd.DataFrame({"y_test_true":y_test, "y_test_pred":y_test_pred})
df_test_preds_full["is_correct"] = df_test_preds_full["y_test_true"] == df_test_preds_full["y_test_pred"]

In [261]:
print(
    "AGE OF SAVED/SACRIFICED PEOPLE AND CULTURE OF DM:\n",
    "train accuracy: ", np.round(df_train_preds_full["is_correct"].mean(), 3), 
    "\ndev accuracy: ", np.round(df_test_preds_full["is_correct"].mean(), 3)
)

AGE OF SAVED/SACRIFICED PEOPLE AND CULTURE OF DM:
 train accuracy:  0.614 
dev accuracy:  0.61


In [262]:
X_names = ["Intercept"] + cols_subset

In [263]:
df_coefs = pd.DataFrame({"featurename": X_names, "coef": result.params, "pval": result.pvalues})

In [264]:
df_coefs

,featurename,coef,pval
0,Intercept,-0.012633,2.277043e-01
1,oldmen_rml,0.071066,4.376141e-68
2,oldwomen_rml,0.113424,2.688110e-39
3,boys_rml,0.300491,7.519873e-255
4,girls_rml,0.344144,0.000000e+00
5,men_rml,0.114070,6.126472e-45
6,women_rml,0.157377,4.862825e-80
7,oldwomen_rmlxEastern,0.014881,3.149753e-01
8,oldwomen_rmlxSouthern,-0.019309,8.690157e-02
9,boys_rmlxEastern,-0.018898,2.083127e-01


Now drop culture dummies

In [265]:
cols_prep = df_subset.columns[~df_subset.columns.str.contains("xEastern") & ~df_subset.columns.str.contains("xSouthern")].to_list()
cols_X = [c for c in cols_prep if not c.startswith("left") and not c.startswith("right")]
cols_X = [c for c in cols_X if c not in ["is_train"]]

In [266]:
cols_X

['Eastern',
 'Southern',
 'oldmen_rml',
 'oldwomen_rml',
 'boys_rml',
 'girls_rml',
 'men_rml',
 'women_rml']

In [267]:
X_train = df_subset.query("is_train")[cols_X].head(n_subset).to_numpy()
X_train = sm.add_constant(X_train)
X_test = df_subset.query("~is_train")[cols_X].head(n_subset).to_numpy()
X_test = sm.add_constant(X_test)
y_train = np.ravel(df_subset.query("is_train")["right_picked"].head(n_subset).to_numpy())
y_test = np.ravel(df_subset.query("~is_train")["right_picked"].head(n_subset).to_numpy())

In [268]:
is_fit = True

In [269]:
model_full = sm.Logit(y_train, X_train)

if fit_model:
    result = model_full.fit()
    joblib.dump(result, "models/mm-sklearn/logreg-age-only.pkl")
else:
    result = joblib.load("models/mm-sklearn/logreg-age-only.pkl")

Optimization terminated successfully.
         Current function value: 0.657898
         Iterations 5


In [270]:
y_test_pred = (result.predict(X_test) > .5).astype(int)
y_train_pred = (result.predict(X_train) > .5).astype(int)
df_train_preds_full = pd.DataFrame({"y_train_true":y_train, "y_train_pred":y_train_pred}) 
df_train_preds_full["is_correct"] = df_train_preds_full["y_train_true"] == df_train_preds_full["y_train_pred"]
df_test_preds_full = pd.DataFrame({"y_test_true":y_test, "y_test_pred":y_test_pred})
df_test_preds_full["is_correct"] = df_test_preds_full["y_test_true"] == df_test_preds_full["y_test_pred"]

In [271]:
print(
    "AGE OF SAVED/SACRIFICED PEOPLE ONLY:\n",
    "train accuracy: ", np.round(df_train_preds_full["is_correct"].mean(), 3), 
    "\ndev accuracy: ", np.round(df_test_preds_full["is_correct"].mean(), 3)
)

AGE OF SAVED/SACRIFICED PEOPLE ONLY:
 train accuracy:  0.614 
dev accuracy:  0.61
